## <font color='blue'>LangChain e LLM Open-Source Para Sistema de Perguntas e Respostas</font>

Este Estudo de Caso demonstra na prática uma série de técnicas, estratégias e ferramentas para construir aplicações inteligentes com LangChain e LLMs. 


Vamos trabalhar com RAG a partir de dados extraídos da web e ao final ainda criaremos um Chatbot baseado em IA especialista na venda de automóveis.

## Instalando e Carregando Pacotes

In [65]:
!pip install -q langchain langchain-classic langchain-core langchain-community langchain-huggingface

^C
ERROR: Operation cancelled by user


In [63]:
# Instala o PyTorch exato com suporte CUDA 12.4
!pip install -q torch==2.10.0

# Instala as restantes bibliotecas com as versões fixadas
!pip install -q transformers==4.54.0 pandas==2.2.2 tokenizers==0.21.2 accelerate==1.9.0 trl==0.20.0 peft==0.16.0 bitsandbytes==0.46.1 scipy==1.16.0 sentencepiece==0.2.0 numba==0.60.0 gradio==5.38.2 datasets==4.0.0

^C
ERROR: Operation cancelled by user


In [3]:
# Imports
import torch
import accelerate
import peft
import bitsandbytes
import transformers
import trl
import datasets
import langchain

In [4]:
# Imports
from trl import SFTTrainer, SFTConfig
from datasets import load_dataset
# from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
# from transformers import pipeline, TrainingArguments
from transformers import (AutoModelForCausalLM,
                          AutoTokenizer,
                          BitsAndBytesConfig,
                          HfArgumentParser,
                          TrainingArguments,
                          pipeline,
                          logging)
from peft import AutoPeftModelForCausalLM, LoraConfig, get_peft_model, prepare_model_for_kbit_training

from langchain_huggingface import HuggingFacePipeline
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

import warnings
warnings.filterwarnings('ignore')

## Carregando o Dataset Para o Instruction Fine-Tuning

https://huggingface.co/datasets/nlpie/Llama2-MedTuned-Instructions

In [5]:
# Carrega o dataset
dataset = load_dataset("nlpie/Llama2-MedTuned-Instructions")

README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001-a8790d88efc2bc(…):   0%|          | 0.00/91.1M [00:00<?, ?B/s]

data/validation-00000-of-00001-b543c64b1(…):   0%|          | 0.00/6.08M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/200252 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/70066 [00:00<?, ? examples/s]

In [6]:
# Selecionamos as linhas para treino do modelo
ft_dados_treino = dataset["train"].select(indices = range(1000))

In [7]:
ft_dados_treino

Dataset({
    features: ['instruction', 'input', 'output', 'source'],
    num_rows: 1000
})

In [8]:
# Selecionamos as linhas para teste do modelo
ft_dados_teste = dataset["train"].select(indices = range(1000, 1200))

In [9]:
ft_dados_teste

Dataset({
    features: ['instruction', 'input', 'output', 'source'],
    num_rows: 200
})

## Compreendendo o Formato dos Dados de Texto

In [10]:
# Vamos visualizar 3 pontos de dados
for i in range(3):
    data = dataset['train'][i]
    print(f"Ponto de Dado {i + 1}:")
    print("Instruction:", data['instruction'])
    print("Input:", data['input'])
    print("Output:", data['output'])
    print("\n-----------------------------\n")

Ponto de Dado 1:
Instruction: In your role as a medical professional, address the user's medical questions and concerns.
Input: My relative suffering from secondary lever cancer ( 4th stage as per Allopathic doctor) and primary is in rectum. He is continuously with 103 to 104 degree F fever. Allpathic doctor suggested chemo only after fever subsidises. Is treatment possible at Lavanya & what is the time scale of recover.
Output: Hi, dairy have gone through your question. I can understand your concern. He has rectal cancer with liver metastasis. It is stage 4 cancer. Surgery is not possible at this stage. Only treatment options are chemotherapy and radiotherapy according to type of cancer. Inspite of all treatment prognosis is poor. Life expectancy is not good. Consult your doctor and plan accordingly. Hope I have answered your question, if you have any doubts then contact me at bit.ly/ Chat Doctor. Thanks for using Chat Doctor. Wish you a very good health.

----------------------------

## Automatizando a Criação dos Prompts Para Treinamento do Modelo

In [11]:
# Define a função que recebe um dicionário chamado sample
def ft_cria_prompt(sample):

    # Define uma string pre_prompt que serve como um modelo para a primeira parte do prompt
    pre_prompt = """[INST]<<SYS>> {instruction}\n"""

    # Concatena o pre_prompt com strings adicionais para formar o prompt completo
    prompt = pre_prompt + "{input}" +"[/INST]"+"\n{output}"

    # Atribui o valor da chave 'instruction' do dicionário sample à variável example_instruction
    example_instruction = sample['instruction']

    # Atribui o valor da chave 'input' do dicionário sample à variável example_input
    example_input = sample['input']

    # Atribui o valor da chave 'output' do dicionário sample à variável example_output
    example_output = sample['output']

    # Cria uma instância de PromptTemplate com o prompt definido anteriormente e as variáveis de entrada
    prompt_template = PromptTemplate(template = prompt,
                                     input_variables = ["instruction", "input", "output"])

    # Utiliza o método format da instância prompt_template para substituir as variáveis
    # no template com os valores específicos
    prompt_unico = prompt_template.format(instruction = example_instruction,
                                          input = example_input,
                                          output = example_output)

    # Retorna o prompt formatado
    return prompt_unico

In [12]:
# Cria a coluna 'text' com o prompt formatado para cada um dos 1.000 exemplos
ft_dados_treino = ft_dados_treino.map(lambda sample: {"text": ft_cria_prompt(sample)})

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [13]:
# Testando a função
prompt = ft_cria_prompt(ft_dados_treino[0])
print(prompt)

[INST]<<SYS>> In your role as a medical professional, address the user's medical questions and concerns.
My relative suffering from secondary lever cancer ( 4th stage as per Allopathic doctor) and primary is in rectum. He is continuously with 103 to 104 degree F fever. Allpathic doctor suggested chemo only after fever subsidises. Is treatment possible at Lavanya & what is the time scale of recover.[/INST]
Hi, dairy have gone through your question. I can understand your concern. He has rectal cancer with liver metastasis. It is stage 4 cancer. Surgery is not possible at this stage. Only treatment options are chemotherapy and radiotherapy according to type of cancer. Inspite of all treatment prognosis is poor. Life expectancy is not good. Consult your doctor and plan accordingly. Hope I have answered your question, if you have any doubts then contact me at bit.ly/ Chat Doctor. Thanks for using Chat Doctor. Wish you a very good health.


In [14]:
# Testando a função
prompt = ft_cria_prompt(ft_dados_teste[0])
print(prompt)

[INST]<<SYS>> In the clinical text, your objective is to identify relationships between medical problems, treatments, and tests. Medical problems are tagged as @problem$, medical tests as @test$, and treatments as @treatment$. Classify the relationship between two entities as one of the following:
Treatment improves medical problem (TrIP)
Treatment worsens medical problem (TrWP)
Treatment causes medical problem (TrCP)
Treatment is administered for medical problem (TrAP)
Treatment is not administered because of medical problem (TrNAP)
Test reveals medical problem (TeRP)
Test conducted to investigate medical problem (TeCP)
Medical problem indicates medical problem (PIP)
No Relations
Include @treatment$ 50 mgs bid , Aricept 10 mgs qhs , @treatment$ 15 mgs bid , Trazodone 100 mgs qhs .[/INST]
No Relations


## Processo de Quantização

In [15]:
# Ativa o carregamento do modelo base com precisão de 4 bits
use_4bit = True

In [16]:
# Define o dtype para o modelo base
bnb_4bit_compute_dtype = "float16"

In [17]:
# Tipo de quantização
bnb_4bit_quant_type = "nf4"

In [18]:
# Desativa a quantização dupla
use_nested_quant = False

In [19]:
# Define o dtype para computação no PyTorch
compute_dtype = getattr(torch, bnb_4bit_compute_dtype)

In [20]:
# Define as configurações
bnb_config = BitsAndBytesConfig(load_in_4bit = use_4bit,
                                bnb_4bit_quant_type = bnb_4bit_quant_type,
                                bnb_4bit_compute_dtype = compute_dtype,
                                bnb_4bit_use_double_quant = use_nested_quant)

In [21]:
# Verifica se a GPU suporta bfloat16
if compute_dtype == torch.float16 and use_4bit:
    major, _ = torch.cuda.get_device_capability()
    if major >= 8:
        print("=" * 80)
        print("A GPU suporta bfloat16. Acelere o treinamento usando bf16=True")
        print("=" * 80)

## Carregando LLM e Tokenizador

https://huggingface.co/NousResearch/Llama-2-7b-chat-hf

In [22]:
# Nome do LLM
nome_llm = "NousResearch/Llama-2-7b-chat-hf"

In [23]:
# Carrega o tokenizador
tokenizer = AutoTokenizer.from_pretrained("NousResearch/Llama-2-7b-chat-hf")

tokenizer_config.json:   0%|          | 0.00/746 [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/21.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/435 [00:00<?, ?B/s]

In [24]:
# Carrega o modelo base com a quantização
modelo = AutoModelForCausalLM.from_pretrained(nome_llm,
                                              quantization_config = bnb_config,
                                              device_map = "auto",
                                              use_cache = False)

config.json:   0%|          | 0.00/583 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/9.98G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/3.50G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

In [25]:
# Usa o token EOS do tokenizador para o pad ao fim de cada sequência
tokenizer.pad_token = tokenizer.eos_token

In [26]:
# Ativa o padding ao final de cada sentença
tokenizer.padding_side = "right"

## Configurando Adaptadores LoRa

In [27]:
# Parâmetros LoRa
ft_peft_config = LoraConfig(r = 8,
                            lora_alpha = 16,
                            lora_dropout = 0.05,
                            bias = "none",
                            task_type = "CAUSAL_LM"
                           )

A quantização representa dados com menos bits, tornando-se uma técnica útil para reduzir o uso de memória e acelerar a inferência, especialmente quando se trata de LLMs.

Depois que um modelo é quantizado, ele normalmente não é treinado DIRETAMENTE para tarefas posteriores porque o treinamento pode ser instável devido à menor precisão dos pesos e ativações. Mas como os métodos PEFT apenas adicionam parâmetros extras treináveis, isso permite treinar um modelo quantizado com um adaptador PEFT na parte superior! Combinar quantização com PEFT pode ser uma boa estratégia para treinar até mesmo os maiores modelos em uma única GPU. Por exemplo, QLoRA é um método que quantiza um modelo em 4 bits e depois o treina com LoRA. Este método permite ajustar um modelo de parâmetros de 65B em uma única GPU de 48GB, por exemplo.

O objetivo do PEFT (Parameter-Efficient Fine-Tuning) é manter a maioria dos parâmetros do modelo pré-treinado fixos e ajustar apenas um pequeno subconjunto de parâmetros para adaptar o modelo a uma tarefa específica.

In [28]:
# Prepara o modelo para treinamento
modelo_ft = prepare_model_for_kbit_training(modelo)

In [29]:
# Junta o modelo quantizado com os adaptadores LoRa
modelo_ft = get_peft_model(modelo_ft, ft_peft_config)

## Parâmetros do Ajuste Fino

In [30]:
output_model = "modelo_ajustado"

In [31]:
# Argumentos de Treino
# ft_training_arguments = TrainingArguments(output_dir = output_model,
#                                           per_device_train_batch_size = 1,
#                                           gradient_accumulation_steps = 4,
#                                           optim = "paged_adamw_32bit",
#                                           learning_rate = 2e-4,
#                                           lr_scheduler_type = "cosine",
#                                           save_strategy = "epoch",
#                                           logging_steps = 10,
#                                           num_train_epochs = 3,
#                                           max_steps = 150,
#                                           fp16 = True)

ft_training_arguments = SFTConfig(output_dir = output_model,
                                   per_device_train_batch_size = 1, # <-- Reduzido de 8 para 2 para caber na GPU
                                   gradient_accumulation_steps = 4, # <-- Aumentado para 4 para manter a velocidade efetiva
                                   optim = "paged_adamw_32bit",
                                   learning_rate = 2e-4,
                                   fp16 = False,
                                   bf16 = True,
                                   max_steps = 150,
                                   lr_scheduler_type = "cosine",
                                   dataset_text_field = "text",
                                   report_to = "none",
                                   dataloader_num_workers = 0
                                 )

In [32]:
# Cria o Trainer
# Otimizado para ajustar modelos pré-treinados com conjuntos de dados menores em tarefas
# de aprendizagem supervisionada.
ft_trainer = SFTTrainer(
                        model = modelo_ft,
                        processing_class = tokenizer,
                        args = ft_training_arguments,
                        train_dataset = ft_dados_treino,
                    )

Adding EOS to train dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


## Ajuste Fino do LLM

In [33]:
%%time
ft_trainer.train()

Step,Training Loss
10,2.968600
20,3.113700
30,3.013800
40,2.787500
50,2.847700
60,3.026800
70,3.043800
80,2.897100
90,3.072400
100,2.902200


CPU times: user 46min 53s, sys: 28min 10s, total: 1h 15min 3s
Wall time: 1h 15min 1s


TrainOutput(global_step=150, training_loss=2.925195083618164, metrics={'train_runtime': 4500.5648, 'train_samples_per_second': 0.133, 'train_steps_per_second': 0.033, 'total_flos': 6783517244375040.0, 'train_loss': 2.925195083618164})

In [34]:
# Salva o modelo
ft_trainer.save_model("modelo_final")

In [35]:
# Merge
merged_model = modelo_ft.merge_and_unload()

## Construindo o Pipeline de Geração de Texto com LangChain

In [36]:
# Cria o pre-prompt com a instrução
pre_prompt = """[INST] <<SYS>>\nAnalyze the question and answer with the best option.\n"""

In [37]:
# Cria o prompt adicionando o input
prompt = pre_prompt + "Here is my question {context}" + "[\INST]"

In [38]:
# Cria o prompt template com LangChain
prompt = PromptTemplate(template = prompt, input_variables = ["context"])

Os pipelines são uma maneira excelente e fácil de usar modelos para inferência. Esses pipelines são objetos que abstraem a maior parte do código complexo da biblioteca, oferecendo uma API simples dedicada a diversas tarefas, incluindo reconhecimento de entidade nomeada, modelagem de linguagem mascarada, análise de sentimento, extração de recursos e resposta a perguntas.

In [39]:
# Cria o objeto pipeline
ft_pipe = pipeline("text-generation",
                    model = merged_model,
                    tokenizer = tokenizer,
                    max_new_tokens = 512,
                    use_cache = False,
                    do_sample = True,
                    pad_token_id = tokenizer.eos_token_id,
                    top_p = 0.7,
                    temperature = 0.5)

Device set to use cuda:0


In [47]:
# Cria o Hugging Face Pipeline
llm_pipeline = HuggingFacePipeline(pipeline = ft_pipe)

## Criando a LLM Chain

In [53]:
!pip install langchain-classic

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [57]:
from langchain_classic.memory import ConversationBufferMemory
from langchain_classic.chains import LLMChain

In [58]:
# Cria a memória
memory = ConversationBufferMemory()

In [59]:
# Cria o LLM Chain
ft_chat_llm_chain = LLMChain(llm = llm_pipeline,
                              prompt = prompt,
                              verbose = False,
                              memory = memory)

/tmp/ipykernel_58/58045777.py:2: LangChainDeprecationWarning: The class `LLMChain` was deprecated in LangChain 0.1.17 and will be removed in 2.0.0. Use `RunnableSequence, e.g., `prompt | llm`` instead.
  ft_chat_llm_chain = LLMChain(llm = llm_pipeline,


## Deploy do Modelo e Uso do Sistema de Perguntas e Respostas

In [60]:
contexto = '''###Question: All of the following provisions are included in the Primary health care according to the Alma Ata declaration except:
###Options:
A. Adequate supply of safe drinking water
B. Nutrition
C. Provision of free medicines
D. Basic sanitation'''

In [ ]:
%%time
ft_chat_llm_chain.predict(context = contexto)

"[INST] <<SYS>>\nAnalyze the question and answer with the best option.\nHere is my question ###Question: All of the following provisions are included in the Primary health care according to the Alma Ata declaration except:\n###Options:\nA. Adequate supply of safe drinking water\nB. Nutrition\nC. Provision of free medicines\nD. Basic sanitation[\\INST]  Great! Let's analyze the question and answer options:\n\nQuestion: All of the following provisions are included in the Primary health care according to the Alma Ata declaration except:\n\nOptions:\nA. Adequate supply of safe drinking water\nB. Nutrition\nC. Provision of free medicines\nD. Basic sanitation\n\nThe Alma Ata declaration is a widely recognized international document that outlines the principles of primary health care. According to the declaration, primary health care should include the following provisions:\n\n* Adequate supply of safe drinking water\n* Nutrition\n* Provision of free medicines\n* Basic sanitation\n\nHowever, the question asks about the exception, which is not included in the Alma Ata declaration. Therefore, the best answer is (D) Basic sanitation.\n\nExplanation:\nThe Alma Ata declaration does not mention anything about basic sanitation. While it does mention the importance of safe drinking water, nutrition, and provision of free medicines, it does not include basic sanitation as one of the essential provisions of primary health care. Therefore, the answer is (D) Basic sanitation."


# Fim